In [11]:
import pandas as pd
import json

In [12]:
file_path = '../data/miband/20260406_8278074507_MiFitness_ams1_data_copy/20260406_8278074507_MiFitness_hlth_center_aggregated_fitness_data.csv'
df_raw = pd.read_csv(file_path)

In [13]:
df_raw

,Uid,Sid,Tag,Key,Time,Value,UpdateTime
0,8278074507,2016481816,daily_report,stress,1764115200,"{""min_stress"":6,""max_stress"":46,""avg_stress"":3...",1764236323
1,8278074507,2016481816,daily_report,stress,1764201600,"{""max_stress"":45,""min_stress"":6,""stress_scale""...",1764320607
2,8278074507,2016481816,daily_report,stress,1764288000,"{""avg_stress"":22,""stress_scale"":{""relax"":104,""...",1764404041
3,8278074507,2016481816,daily_report,stress,1764374400,"{""max_stress"":44,""avg_stress"":21,""min_stress"":...",1764498229
4,8278074507,2016481816,daily_report,stress,1764460800,"{""avg_stress"":22,""min_stress"":10,""stress_scale...",1764704127
...,...,...,...,...,...,...,...
5095,8278074507,default,daily_fitness,goal,1775088000,"{""goal_items"":[{""field"":1,""target_value"":10000...",1775210499
5096,8278074507,default,daily_fitness,goal,1775174400,"{""date_time"":1775174400,""goal_items"":[{""achiev...",1775295031
5097,8278074507,default,daily_fitness,goal,1775260800,"{""date_time"":1775260800,""goal_items"":[{""field""...",1775382388
5098,8278074507,default,daily_fitness,goal,1775347200,"{""date_time"":1775347200,""goal_items"":[{""achiev...",1775467586


In [14]:
df_stress = df_raw[(df_raw['Key'] == 'stress') & (df_raw['Tag'] == 'daily_report')].copy()

# Parse JSON values and normalize into separate columns
stress_data_list = [json.loads(val) for val in df_stress['Value']]
df_stress_normalized = pd.json_normalize(stress_data_list)

# Flatten nested stress_scale columns
stress_scale_cols = [col for col in df_stress_normalized.columns if col.startswith('stress_scale.')]
for col in stress_scale_cols:
    new_col_name = col.replace('stress_scale.', '')
    df_stress_normalized[new_col_name] = df_stress_normalized[col]
    df_stress_normalized = df_stress_normalized.drop(col, axis=1)

# Combine with original metadata (Time, Uid, etc.)
df_stress_result = df_stress[['Time']].reset_index(drop=True).join(df_stress_normalized)

# Drop 'severe' and 'moderate' columns
df_stress_result = df_stress_result.drop(['severe', 'moderate'], axis=1)

df_stress_result


,Time,min_stress,max_stress,avg_stress,relax,mild
0,1764115200,6,46,30,12,35
1,1764201600,6,45,22,101,40
2,1764288000,11,43,22,104,47
3,1764374400,10,44,21,99,30
4,1764460800,10,42,22,123,56
...,...,...,...,...,...,...
289,1763769600,30,39,36,0,9
290,1763856000,7,42,24,82,54
291,1763942400,12,37,18,126,15
292,1764028800,11,43,20,92,30


In [15]:
# Convert Time and UpdateTime columns from epoch to datetime
df_stress_result['Date'] = pd.to_datetime(df_stress_result['Time'], unit='s')

df_stress_result = df_stress_result.drop(columns=['Time'])

df_stress_result

,min_stress,max_stress,avg_stress,relax,mild,Date
0,6,46,30,12,35,2025-11-26
1,6,45,22,101,40,2025-11-27
2,11,43,22,104,47,2025-11-28
3,10,44,21,99,30,2025-11-29
4,10,42,22,123,56,2025-11-30
...,...,...,...,...,...,...
289,30,39,36,0,9,2025-11-22
290,7,42,24,82,54,2025-11-23
291,12,37,18,126,15,2025-11-24
292,11,43,20,92,30,2025-11-25


In [16]:
import os

output_dir = '../data/processed_data'
output_path = os.path.join(output_dir, 'miband_stress_processed.csv')
df_stress_result.to_csv(output_path, index=False)